In [1]:
import torch
from torchvision.datasets import MNIST, CIFAR10, USPS, QMNIST
import numpy as np
import torch.nn.functional as F
from torch.linalg import matrix_norm

In [2]:
#SR_DISTANCE
print("SR_DISTANCE")

mnist = MNIST(root='data', train=True, download=True)
cifar = CIFAR10(root='data', train=True, download=True)
usps = USPS(root='data', train=True, download=True)

data_np_mnist = mnist.data[:5000].numpy().astype(np.float32) / 255.0  # Use subset for speed
data_np_mnist = data_np_mnist[:, None, :, :]
mnist_tensor = torch.from_numpy(data_np_mnist).float()
mnist_tensor = F.interpolate(mnist_tensor, size=(32, 32), mode='bilinear', align_corners=False)
mnist_tensor = mnist_tensor.repeat(1, 3, 1, 1)
data_torch_mnist = mnist_tensor.detach().cpu()
data_torch_mnist_resized = F.interpolate(data_torch_mnist, size=(64, 64), mode='bilinear', align_corners=False)
data_torch_mnist_flat = data_torch_mnist_resized.view(5000, -1)
data_flattened_mnist = data_torch_mnist_flat.flatten()
final_mnist = data_flattened_mnist[:3072*3072]
data_np_mnist = final_mnist.view(3072, 3072).numpy()

data_np_cifar = cifar.data[:5000].astype(np.float32) / 255.0  # Use subset for speed
data_torch_cifar = torch.from_numpy(data_np_cifar.transpose(0, 3, 1, 2))
data_torch_cifar_resized = F.interpolate(data_torch_cifar, size=(64, 64), mode='bilinear', align_corners=False)
data_torch_cifar_flat = data_torch_cifar_resized.view(5000, -1)
data_flattened_cifar = data_torch_cifar_flat.flatten()
final_cifar = data_flattened_cifar[:3072*3072]
data_np_cifar = final_cifar.view(3072, 3072).numpy()

usps = USPS(root='data', train=True, download=True)
data_np_usps = usps.data.astype(np.float32) / 255.0
usps_tensor = torch.from_numpy(data_np_usps[:, None, :, :])
usps_tensor = F.interpolate(usps_tensor, size=(32, 32), mode='bilinear', align_corners=False)
usps_tensor = usps_tensor.repeat(1, 3, 1, 1)
usps_tensor = usps_tensor.view(7291, -1)
data_flattened_usps = usps_tensor.flatten()
target_size = 3072 * 3072
if data_flattened_usps.numel() < target_size:
    data_flattened_usps = F.pad(data_flattened_usps, (0, target_size - data_flattened_usps.numel()))
else:
    data_flattened_usps = data_flattened_usps[:target_size]
final_usps = data_flattened_usps.view(3072, 3072)
data_np_usps = final_usps.cpu().numpy()

qmnist = QMNIST(root='data', train=True, download=True)
data_np_qmnist = qmnist.data.numpy().astype(np.float32) / 255.0
data_np_qmnist = data_np_qmnist[:, None, :, :]  
qmnist_tensor = torch.from_numpy(data_np_qmnist)
qmnist_tensor = F.interpolate(qmnist_tensor, size=(32, 32), mode='bilinear', align_corners=False)
qmnist_tensor = qmnist_tensor.repeat(1, 3, 1, 1)[:5000]
data_torch_qmnist_flat = qmnist_tensor.view(5000, -1)
data_flattened_qmnist = data_torch_qmnist_flat.flatten()
final_qmnist = data_flattened_qmnist[:3072*3072]
data_np_qmnist = final_qmnist.view(3072, 3072).numpy()

print(data_np_cifar.shape, data_np_mnist.shape, data_np_usps.shape, data_np_qmnist.shape)

SR_DISTANCE
(3072, 3072) (3072, 3072) (3072, 3072) (3072, 3072)


In [3]:
data_np_cifar_inv = np.linalg.pinv(data_np_cifar)
data_np_usps_inv = np.linalg.pinv(data_np_usps)
data_np_qmnist_inv = np.linalg.pinv(data_np_qmnist)
print(data_np_cifar_inv.shape, data_np_usps_inv.shape)

(3072, 3072) (3072, 3072)


In [4]:
transformation_matrix_cifar = data_np_mnist @ data_np_cifar_inv
tranformation_matrix_usps = data_np_mnist @ data_np_usps_inv
transformation_matrix_qmnist = data_np_mnist @ data_np_qmnist_inv

In [5]:
I_cifar = np.eye(3072)
I_usps = np.eye(3072)
I_qmnist = np.eye(3072)

norm_bef_cifar = transformation_matrix_cifar - I_cifar
norm_bef_usps = tranformation_matrix_usps - I_usps
norm_bef_qmnist = transformation_matrix_qmnist - I_qmnist


norm_af_cifar = np.linalg.matrix_norm(norm_bef_cifar, ord=-1)
nor_af_usps = np.linalg.matrix_norm(norm_bef_usps, ord=-1)
norm_af_qmnist = np.linalg.matrix_norm(norm_bef_qmnist, ord=-1)
print("CIFAR:", norm_af_cifar)
print("USPS:", nor_af_usps)
print("QM-NIST:", norm_af_qmnist)

CIFAR: 1291.193175064458
USPS: 96799266.57108426
QM-NIST: 6.138094532477243
